In [ ]:
import json
import re
from datasets import load_dataset

# 1) Load the dataset
ds = load_dataset("Goedel-LM/Lean-workbook-proofs")

def parse_lean_proof(full_text: str):
    """
    Attempt to split 'full_text' into:
      - header (imports, set_option, open ...)
      - informal_prefix (the /- ... -/ comment describing the statement)
      - formal_statement (the theorem line)
      - completion (the proof steps after 'theorem ... := by')
    """

    header = ""
    informal_prefix = ""
    formal_statement = ""
    completion = ""

    # --- 1) Extract header: everything from the start until we see either '/-' or 'theorem' ---
    #
    # Explanation:
    #   We try to match from the beginning ^(.*?) 
    #   up to (but not including) the first occurrence of '/-' or 'theorem' 
    #   using a capturing group (.*?) in DOTALL mode.
    header_pattern = r'^(.*?)(?=/-|theorem)'
    header_match = re.search(header_pattern, full_text, flags=re.DOTALL)
    if header_match:
        header = header_match.group(1).strip()

    # --- 2) Extract the informal prefix inside /- ... -/ ---
    #
    # Explanation:
    #   We match /- then lazily capture everything up to the next -/
    #   in DOTALL mode to handle multiline comments.
    informal_prefix_pattern = r'/-(.*?)-/'
    ip_match = re.search(informal_prefix_pattern, full_text, flags=re.DOTALL)
    if ip_match:
        # Reconstruct the entire comment with /- and -/
        informal_prefix = f"/-{ip_match.group(1)}-/".strip()

    # --- 3) Extract the formal statement, i.e. "theorem ... := by" ---
    #
    # Explanation:
    #   This tries to capture from "theorem" up through ":= by" 
    #   (which is common in Lean code).
    formal_statement_pattern = r'(theorem\s.*?:\s.*?:=\s*by)'
    fs_match = re.search(formal_statement_pattern, full_text, flags=re.DOTALL)
    if fs_match:
        formal_statement = fs_match.group(1).strip()

    # --- 4) Everything that follows the formal statement is the "completion" ---
    if fs_match:
        completion_start = fs_match.end(1)
        completion = full_text[completion_start:].strip()

    return header, informal_prefix, formal_statement, completion

# 2) Write each processed record to a JSONL file.
#    Adjust "train" below as needed (you might do this for test/validation too).
output_filename = "./lean_workbook_split.jsonl"
with open(output_filename, "w", encoding="utf-8") as f_out:
    for sample in ds["train"]:
        problem_id = sample["problem_id"]
        full_proof = sample["full_proof"]

        # Parse out the desired parts
        header, informal_prefix, formal_statement, completion = parse_lean_proof(full_proof)

        # Create the record in the desired schema
        record = {
            "name": problem_id,
            "split": "train",
            "informal_prefix": informal_prefix,
            "formal_statement": formal_statement,
            "header": header,
            "completion": completion
        }

        # Write out as JSON
        f_out.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Done! Wrote split data to {output_filename}.")